In [1]:
import os, re, time, getpass, sqlite3
import pandas as pd
from google import genai

con = sqlite3.connect("file:bengaluru_restaurants.db?mode=ro", uri=True)

def get_schema():
    out = []
    for (t,) in con.execute("SELECT name FROM sqlite_master WHERE type='table'"):
        cols = con.execute(f"PRAGMA table_info({t})").fetchall()
        out.append(f"{t}(" + ", ".join(f"{c[1]} {c[2]}" for c in cols) + ")")
    return "\n".join(out)

def run_sql(query):
    q = query.strip().rstrip(";")
    if not re.match(r"^(select|with)\b", q, re.I):
        return "Error: only SELECT queries are allowed"
    if re.search(r"\b(insert|update|delete|drop|alter|create|attach|pragma|replace)\b", q, re.I):
        return "Error: this query contains a blocked keyword"
    try:
        return pd.read_sql(q, con)
    except Exception as e:
        return f"Error: {e}"

os.environ["GEMINI_API_KEY"] = getpass.getpass("Paste your Gemini API key: ")
client = genai.Client()

CANDIDATES = ["gemini-3.8-flash", "gemini-3.7-flash", "gemini-3.6-flash",
              "gemini-3.5-flash", "gemini-3.5-flash-lite", "gemini-3.1-flash-lite"]
MODEL = None

def ask_llm(prompt):
    global MODEL
    order = ([MODEL] if MODEL else []) + [m for m in CANDIDATES if m != MODEL]
    last = None
    for m in order:
        for attempt in range(2):
            try:
                text = client.models.generate_content(model=m, contents=prompt).text
                MODEL = m
                return text
            except Exception as e:
                last = e
                if "503" in str(e) or "429" in str(e):
                    time.sleep(4)
                    continue
                break
    raise RuntimeError(f"All models failed. Last error: {str(last)[:200]}")

print(ask_llm("Reply with the single word: ready"), "| using", MODEL)

Paste your Gemini API key:  ········


ready | using gemini-3.6-flash


In [2]:
SCHEMA = get_schema()

def make_sql(question):
    prompt = f"""You write SQLite queries for a Bengaluru restaurants database.

Tables:
{SCHEMA}

Rules:
- Return ONLY one SQL query. No explanation, no markdown.
- Use only SELECT.
- restaurants has one row per restaurant. rate can be NULL (unrated).
- Cuisines are in restaurant_cuisines. Join on restaurant_id.
- When ranking by average rating, only include groups with at least 30 restaurants.
- Add LIMIT 20 unless the question asks otherwise.

Question: {question}"""
    sql = ask_llm(prompt).strip()
    sql = re.sub(r"^```(?:sql)?|```$", "", sql, flags=re.M).strip()
    return sql

In [3]:
q = "Which 5 areas have the most restaurants?"
sql = make_sql(q)
print(sql)
run_sql(sql)

SELECT location, COUNT(*) FROM restaurants GROUP BY location ORDER BY COUNT(*) DESC LIMIT 5;


,location,COUNT(*)
0,Whitefield,882
1,Electronic City,729
2,BTM,728
3,HSR,705
4,Marathahalli,684


In [4]:
def clean(sql):
    return re.sub(r"^```(?:sql)?|```$", "", sql.strip(), flags=re.M).strip()

def explain(question, sql, result):
    prompt = f"""Question: {question}
SQL used: {sql}
Result:
{result.head(20).to_string(index=False)}

Answer the question in 1 to 3 plain sentences using only this result. Do not add facts."""
    return ask_llm(prompt).strip()

def ask(question, max_fixes=2):
    sql = make_sql(question)
    result = run_sql(sql)
    fixes = 0
    while not isinstance(result, pd.DataFrame) and fixes < max_fixes:
        fixes += 1
        fix = f"""This SQLite query failed.
Schema:
{SCHEMA}
Question: {question}
Query: {sql}
Error: {result}
Return ONLY the corrected SQL query."""
        sql = clean(ask_llm(fix))
        result = run_sql(sql)
    if not isinstance(result, pd.DataFrame):
        return {"question": question, "sql": sql, "result": None,
                "answer": "Could not answer: " + str(result), "fixes": fixes}
    return {"question": question, "sql": sql, "result": result,
            "answer": explain(question, sql, result), "fixes": fixes}

def show(question):
    out = ask(question)
    print("SQL:", out["sql"])
    if out["result"] is not None:
        display(out["result"])
    print("\nAnswer:", out["answer"])

In [5]:
show("Which cuisines have the highest average rating among cuisines with at least 100 restaurants?")

SQL: SELECT rc.cuisine, AVG(r.rate) AS avg_rating FROM restaurant_cuisines rc JOIN restaurants r ON rc.restaurant_id = r.restaurant_id GROUP BY rc.cuisine HAVING COUNT(r.restaurant_id) >= 100 ORDER BY avg_rating DESC LIMIT 20


,cuisine,avg_rating
0,European,4.160577
1,Asian,4.093407
2,BBQ,3.991270
3,American,3.975377
4,Finger Food,3.930818
5,Mexican,3.930612
6,Salad,3.916201
7,Continental,3.888429
8,Italian,3.884032
9,Thai,3.849189



Answer: Among cuisines with at least 100 restaurants, European has the highest average rating at 4.16, followed by Asian at 4.09 and BBQ at 3.99. Other top-ranking cuisines include American (3.98), Finger Food (3.93), and Mexican (3.93).


In [6]:
TESTS = [
("How many restaurants are there in total?", "SELECT COUNT(*) FROM restaurants"),
("What is the average rating of all rated restaurants? Round to 2 decimals.", "SELECT ROUND(AVG(rate),2) FROM restaurants"),
("Which 5 areas have the most restaurants?", "SELECT location FROM restaurants GROUP BY location ORDER BY COUNT(*) DESC LIMIT 5"),
("What is the average cost for two across all restaurants? Round to a whole number.", "SELECT ROUND(AVG(cost_for_two)) FROM restaurants"),
("How many restaurants offer online ordering?", "SELECT COUNT(*) FROM restaurants WHERE online_order='Yes'"),
("How many restaurants allow table booking?", "SELECT COUNT(*) FROM restaurants WHERE book_table='Yes'"),
("Which 5 cuisines are the most common?", "SELECT cuisine FROM restaurant_cuisines GROUP BY cuisine ORDER BY COUNT(*) DESC LIMIT 5"),
("How many restaurants cost more than 1000 for two?", "SELECT COUNT(*) FROM restaurants WHERE cost_for_two > 1000"),
("Which restaurant has the highest cost for two?", "SELECT name FROM restaurants ORDER BY cost_for_two DESC LIMIT 1"),
("Which area has the highest average rating among areas with at least 50 restaurants?", "SELECT location FROM restaurants WHERE rate IS NOT NULL GROUP BY location HAVING COUNT(*)>=50 ORDER BY AVG(rate) DESC LIMIT 1"),
("How many North Indian restaurants are there?", "SELECT COUNT(*) FROM restaurant_cuisines WHERE cuisine='North Indian'"),
("How many restaurants are in Whitefield?", "SELECT COUNT(*) FROM restaurants WHERE location='Whitefield'"),
("How many restaurants have no rating?", "SELECT COUNT(*) FROM restaurants WHERE rate IS NULL"),
("Which 3 areas have the most North Indian restaurants?", "SELECT r.location FROM restaurants r JOIN restaurant_cuisines c ON c.restaurant_id=r.restaurant_id WHERE c.cuisine='North Indian' GROUP BY r.location ORDER BY COUNT(*) DESC LIMIT 3"),
("How many restaurants cost under 300 for two?", "SELECT COUNT(*) FROM restaurants WHERE cost_for_two < 300"),
("Which cuisine has the highest average rating among cuisines with at least 100 restaurants?", "SELECT c.cuisine FROM restaurant_cuisines c JOIN restaurants r ON r.restaurant_id=c.restaurant_id WHERE r.rate IS NOT NULL GROUP BY c.cuisine HAVING COUNT(*)>=100 ORDER BY AVG(r.rate) DESC LIMIT 1"),
]

def is_correct(agent_df, gold_df):
    if not isinstance(agent_df, pd.DataFrame) or agent_df.empty:
        return False
    if gold_df.shape == (1, 1):
        g = gold_df.iloc[0, 0]
        for v in agent_df.iloc[0].tolist():
            try:
                if isinstance(g, str):
                    if str(v) == g:
                        return True
                elif abs(float(v) - float(g)) < 0.01:
                    return True
            except Exception:
                pass
        return False
    return agent_df.iloc[:, 0].tolist() == gold_df.iloc[:, 0].tolist()

def ask_sql(question, max_fixes=2):
    sql = make_sql(question)
    result = run_sql(sql)
    fixes = 0
    while not isinstance(result, pd.DataFrame) and fixes < max_fixes:
        fixes += 1
        fix = f"""This SQLite query failed.
Schema:
{SCHEMA}
Question: {question}
Query: {sql}
Error: {result}
Return ONLY the corrected SQL query."""
        sql = clean(ask_llm(fix))
        result = run_sql(sql)
    return sql, result, fixes

rows = []
for q, gold_sql in TESTS:
    sql, result, fixes = ask_sql(q)
    ok = is_correct(result, run_sql(gold_sql))
    rows.append({"question": q, "correct": ok, "fixes": fixes, "agent_sql": sql})
    print("PASS" if ok else "FAIL", "|", q)
    time.sleep(1)

report = pd.DataFrame(rows)
print(f"\nAccuracy: {report.correct.sum()}/{len(report)} = {report.correct.mean():.0%}")
report.to_csv("eval_results.csv", index=False)

PASS | How many restaurants are there in total?
PASS | What is the average rating of all rated restaurants? Round to 2 decimals.
PASS | Which 5 areas have the most restaurants?
PASS | What is the average cost for two across all restaurants? Round to a whole number.
PASS | How many restaurants offer online ordering?
PASS | How many restaurants allow table booking?
PASS | Which 5 cuisines are the most common?
PASS | How many restaurants cost more than 1000 for two?
PASS | Which restaurant has the highest cost for two?
PASS | Which area has the highest average rating among areas with at least 50 restaurants?
PASS | How many North Indian restaurants are there?
PASS | How many restaurants are in Whitefield?
PASS | How many restaurants have no rating?
FAIL | Which 3 areas have the most North Indian restaurants?
PASS | How many restaurants cost under 300 for two?
PASS | Which cuisine has the highest average rating among cuisines with at least 100 restaurants?

Accuracy: 15/16 = 94%


In [8]:
for _, r in report[~report.correct].iterrows():
    print("Q:", r.question)
    print("SQL:", r.agent_sql)
    res = run_sql(r.agent_sql)
    print(res.head(5) if isinstance(res, pd.DataFrame) else res)
    print("-" * 60)

Q: Which 3 areas have the most North Indian restaurants?
SQL: SELECT r.listing_area FROM restaurants r JOIN restaurant_cuisines rc ON r.restaurant_id = rc.restaurant_id WHERE rc.cuisine = 'North Indian' GROUP BY r.listing_area ORDER BY count(*) DESC LIMIT 3
        listing_area
0                BTM
1        Brookefield
2  Bannerghatta Road
------------------------------------------------------------


In [9]:
SCHEMA = get_schema() + """

Column notes:
- restaurants.location: the neighbourhood or area of the restaurant (for example Whitefield, BTM). When a question says "area" or "neighbourhood", use location.
- restaurants.listing_area: a broader Zomato listing region. Do NOT use it unless the question mentions "listing".
- restaurants.listing_type: the Zomato listing category (Buffet, Delivery, Dine-out). It is not the cuisine.
- restaurants.rate: rating out of 5, NULL if unrated.
- restaurants.cost_for_two: approximate cost for two people in rupees.
- restaurants.online_order and restaurants.book_table: 'Yes' or 'No'.
- restaurants.rest_type: type of place (Quick Bites, Casual Dining, Cafe).
- restaurant_cuisines: one row per restaurant per cuisine.
"""
print(SCHEMA)

restaurants(restaurant_id INTEGER, address TEXT, name TEXT, online_order TEXT, book_table TEXT, rate REAL, votes INTEGER, location TEXT, rest_type TEXT, cuisines TEXT, cost_for_two REAL, listing_type TEXT, listing_area TEXT)
restaurant_cuisines(restaurant_id INTEGER, cuisine TEXT)

Column notes:
- restaurants.location: the neighbourhood or area of the restaurant (for example Whitefield, BTM). When a question says "area" or "neighbourhood", use location.
- restaurants.listing_area: a broader Zomato listing region. Do NOT use it unless the question mentions "listing".
- restaurants.listing_type: the Zomato listing category (Buffet, Delivery, Dine-out). It is not the cuisine.
- restaurants.rate: rating out of 5, NULL if unrated.
- restaurants.cost_for_two: approximate cost for two people in rupees.
- restaurants.online_order and restaurants.book_table: 'Yes' or 'No'.
- restaurants.rest_type: type of place (Quick Bites, Casual Dining, Cafe).
- restaurant_cuisines: one row per restaurant pe

In [10]:
HOLDOUT = [
("How many restaurants are in BTM?", "SELECT COUNT(*) FROM restaurants WHERE location='BTM'"),
("What is the average cost for two in Indiranagar? Round to a whole number.", "SELECT ROUND(AVG(cost_for_two)) FROM restaurants WHERE location='Indiranagar'"),
("Which 3 cuisines have the most restaurants in HSR?", "SELECT c.cuisine FROM restaurants r JOIN restaurant_cuisines c ON c.restaurant_id=r.restaurant_id WHERE r.location='HSR' GROUP BY c.cuisine ORDER BY COUNT(*) DESC LIMIT 3"),
("How many restaurants have a rating of 4.5 or higher?", "SELECT COUNT(*) FROM restaurants WHERE rate >= 4.5"),
("Which area has the most restaurants that offer table booking?", "SELECT location FROM restaurants WHERE book_table='Yes' GROUP BY location ORDER BY COUNT(*) DESC LIMIT 1"),
("What percentage of restaurants offer online ordering? Round to 1 decimal.", "SELECT ROUND(100.0*SUM(online_order='Yes')/COUNT(*),1) FROM restaurants"),
("Which restaurant has the most votes?", "SELECT name FROM restaurants ORDER BY votes DESC LIMIT 1"),
("How many different cuisines are there?", "SELECT COUNT(DISTINCT cuisine) FROM restaurant_cuisines"),
("Which area has the highest average cost for two among areas with at least 30 restaurants?", "SELECT location FROM restaurants GROUP BY location HAVING COUNT(*)>=30 ORDER BY AVG(cost_for_two) DESC LIMIT 1"),
("How many restaurants serve both North Indian and Chinese cuisine?", "SELECT COUNT(*) FROM (SELECT restaurant_id FROM restaurant_cuisines WHERE cuisine IN ('North Indian','Chinese') GROUP BY restaurant_id HAVING COUNT(DISTINCT cuisine)=2)"),
]

rows2 = []
for q, gold_sql in HOLDOUT:
    sql, result, fixes = ask_sql(q)
    ok = is_correct(result, run_sql(gold_sql))
    rows2.append({"question": q, "correct": ok, "fixes": fixes, "agent_sql": sql})
    print("PASS" if ok else "FAIL", "|", q)
    time.sleep(1)

report2 = pd.DataFrame(rows2)
print(f"\nHoldout accuracy: {report2.correct.sum()}/{len(report2)} = {report2.correct.mean():.0%}")
report2.to_csv("eval_results_holdout.csv", index=False)

PASS | How many restaurants are in BTM?
PASS | What is the average cost for two in Indiranagar? Round to a whole number.
PASS | Which 3 cuisines have the most restaurants in HSR?
PASS | How many restaurants have a rating of 4.5 or higher?
PASS | Which area has the most restaurants that offer table booking?
PASS | What percentage of restaurants offer online ordering? Round to 1 decimal.
PASS | Which restaurant has the most votes?
PASS | How many different cuisines are there?
PASS | Which area has the highest average cost for two among areas with at least 30 restaurants?
PASS | How many restaurants serve both North Indian and Chinese cuisine?

Holdout accuracy: 10/10 = 100%
